<a href="https://colab.research.google.com/github/sukhadaghewari/tropicalcyclone-ri-bench/blob/main/notebooks/05_make_storm_identification_prompts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Storm-identification probe

Can a model recognise which real storm it is looking at, just from the storm history? If it can, it may be recalling what the storm did instead of predicting.

This notebook builds identification prompts for every case of the development set (40 storms, 720 cases) in two versions, and for every case of the test set (100 storms, 1,765 cases) without location only:

| File | Storm history shows |
|---|---|
| `model_inputs/storm_identification/dev/storm_identification_dev_2016_text_only_no_location.jsonl.gz` | exactly the RI prompt's 16 features (no `lat`, `lon`, `abs_lat`) |
| `model_inputs/storm_identification/dev/storm_identification_dev_2016_text_only_with_location.jsonl.gz` | the same, plus `lat`, `lon` and `abs_lat` |
| `model_inputs/storm_identification/test/storm_identification_test_2017-2025_text_only_no_location.jsonl.gz` | exactly the RI prompt's 16 features, for the test cases |

Each prompt asks for the storm's name, season and basin, a confidence and a short reason, and allows "unknown" (template: `prompt/storm_identification.md`). The history in each prompt is the same growing history as in the RI prompts, so the `no_location` version measures how identifiable the RI inputs themselves are, and the `with_location` version shows how much location adds.

The true answers are in each record's `name`, `season` and `basin`. Score name **and** season together, because names are reused across years and basins.


## Setup

The next cell works out the repo folder and makes the code in `src/` importable, so the notebook runs the same way locally and in Colab. In Colab it first clones this repo and moves into it.

In [ ]:
# Setup: in Colab, clone the repo and move into it. Then put src/ on the import path.
import os, subprocess, sys
from pathlib import Path

REPO = "tropicalcyclone-ri-bench"
if "google.colab" in sys.modules:
    try:
        os.getcwd()
    except FileNotFoundError:              # the folder we were in was deleted (e.g. an old clone): start again
        os.chdir("/content")
if "google.colab" in sys.modules and os.path.basename(os.getcwd()) != REPO:
    if not os.path.exists(REPO):
        subprocess.run(["git", "clone", f"https://github.com/sukhadaghewari/{REPO}.git"], check=True)
    os.chdir(REPO)
if "google.colab" in sys.modules:
    # A clone from an earlier run survives "Restart session": bring it up to date with GitHub.
    if subprocess.run(["git", "pull", "--ff-only"]).returncode != 0:
        print("WARNING: could not update the repo with git pull; using the files as they are.")

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()   # repo root, locally or in Colab
sys.path.insert(0, str(ROOT / "src"))

In [ ]:
import pandas as pd

from prompts import (prepare_storm, read_records, build_identify_prompt,
                     write_identify_cases, parse_identify_response)

## The cases

The cases come straight from the RI case files written by `02_make_prompt_files.ipynb`, so each probe file covers exactly the same cases, in the same order, with the same `case_id`s. The storm histories (including `lat` and `lon`) are rebuilt from the data CSV.

In [ ]:
# keep_default_na=False: otherwise pandas reads the North Atlantic basin code "NA" as a missing value.
df_all = pd.read_csv(ROOT / "data" / "ibtracs_era5_sat_2016_2025.csv.gz", keep_default_na=False, na_values=[""])

SETS = [  # (split, RI case file, prefix of the probe files, probe versions)
    ("dev", "dev_2016_20ri_20nori_text_only", "storm_identification_dev_2016_text_only", ["no_location", "with_location"]),
    ("test", "test_2017-2025_50ri_50nori_text_only", "storm_identification_test_2017-2025_text_only", ["no_location"]),
]


def load_set(ri_name):
    """The RI cases of one set, and the same cases rebuilt from the data CSV (with lat and lon)."""
    path = ROOT / "model_inputs" / "ri_prediction" / ri_name.split("_")[0] / f"{ri_name}.jsonl.gz"
    assert path.exists(), f"{path.relative_to(ROOT)} not found. Run 02_make_prompt_files.ipynb, or update this clone with git pull."
    cases = read_records(path)
    sids = list(dict.fromkeys(c["sid"] for c in cases))          # storms in case-file order
    group = {c["sid"]: c["storm_group"] for c in cases}
    df_eval = pd.concat([prepare_storm(df_all, sid) for sid in sids], ignore_index=True)
    df_eval["storm_group"] = df_eval["sid"].map(group)
    ids = [f"{r.sid}_{r.obs_index:03d}" for r in df_eval.itertuples()]
    assert ids == [c["case_id"] for c in cases], f"rebuilt cases don't match {path.name}"
    print(f"{len(df_eval)} cases from {len(sids)} storms, matching {path.name}")
    return cases, df_eval


loaded = {split: load_set(ri_name) for split, ri_name, _, _ in SETS}

## Check one case

For one case, the `no_location` history must be exactly the history in the RI prompt, and the `with_location` history the same plus `lat`, `lon` and `abs_lat`.

In [ ]:
import json
import re

HISTORY_RE = re.compile(r"<storm_history>\n(.*?)\n</storm_history>", re.DOTALL)
history = lambda prompt: json.loads(HISTORY_RE.search(prompt).group(1))

for split, (cases, df_eval) in loaded.items():
    i = 5
    rows = df_eval.loc[i - df_eval.loc[i, "obs_index"] : i]
    h_ri = history(cases[i]["prompt"])
    h_no = history(build_identify_prompt(rows, "no_location"))
    h_with = history(build_identify_prompt(rows, "with_location"))
    assert h_no == h_ri
    assert [{k: v for k, v in r.items() if k not in ("lat", "lon", "abs_lat")} for r in h_with] == h_ri
    print(split, cases[i]["case_id"], "| last history row with location:", h_with[-1])

## Write the probe files

One record per case: `case_id`, `split`, `probe` (`identify_no_location` or `identify_with_location`), the true answer and other metadata (`sid`, `name`, `season`, `basin`, `iso_time`, `obs_index`, `hours_since_start`, `storm_group`), and the `prompt`. Send the model only the `prompt`.

In [ ]:
for split, _, prefix, variants in SETS:
    df_eval = loaded[split][1]
    for variant in variants:
        path = ROOT / "model_inputs" / "storm_identification" / split / f"{prefix}_{variant}.jsonl.gz"
        write_identify_cases(df_eval, path, split, variant)
        print(f"saved {len(df_eval)} {split} cases to {path.relative_to(ROOT)} ({path.stat().st_size / 1e6:.1f} MB compressed)")

## Using the answers

Parse each answer with `parse_identify_response`: it returns `storm_name` (upper case, or "unknown"), `season`, `basin`, `confidence` and `reason`, and raises `ValueError` for anything malformed. A sketch of the loop and the scoring:

In [ ]:
# cases = read_records(ROOT / "model_inputs" / "storm_identification" / "dev" / "storm_identification_dev_2016_text_only_no_location.jsonl.gz")
# results = []
# for case in cases:
#     try:
#         out = parse_identify_response(call_llm(case["prompt"]))
#     except ValueError as e:
#         out = {"parse_error": str(e)}
#     results.append({"case_id": case["case_id"], "probe": case["probe"], **out})
#
# r = pd.DataFrame(cases).drop(columns="prompt").merge(pd.DataFrame(results), on=["case_id", "probe"])
# r["name_correct"] = r["storm_name"] == r["name"]
# r["name_and_season_correct"] = r["name_correct"] & (r["season_y"] == r["season_x"])
# r["basin_correct"] = r["basin_y"] == r["basin_x"]